## Conexión al robot

In [ ]:
import time
import numpy as np
from datetime import datetime
from pymycobot.mycobot import MyCobot

PORT = '/dev/ttyUSB0'   # cambiar a /dev/ttyTHS1 si usas UART de Jetson
BAUD = 1000000

mc = MyCobot(PORT, BAUD)
mc.power_on()
time.sleep(1)

print(f'Conectado en {PORT}')
print(f'Ángulos actuales: {mc.get_angles()}')

---
## Pregunta 1 — Definir las poses clave del ciclo

**Enunciado:** Definir `init_pose`, `pick_upper`, `pick_grasp`, `place_upper` y `place_grasp` como vectores de ángulos articulares reales del robot.

**Solución:** Se mueve el brazo manualmente a cada posición y se registra con `mc.get_angles()`. Las poses quedan en `None` hasta ser capturadas.

> Ejecuta cada sub-celda después de mover el brazo a la posición indicada.

In [ ]:
# Pose 1/5 — init_pose
# El robot debe estar en su posición de reposo segura.
# Normalmente es [0,0,0,0,0,0] pero se verifica con el robot real.

mc.send_angles([0, 0, 0, 0, 0, 0], 30)
time.sleep(3)

init_pose = mc.get_angles()
print(f'init_pose  = {init_pose}')

In [ ]:
# Pose 2/5 — pick_upper
# Mueve el brazo manualmente hasta quedar SOBRE el objeto (sin tocarlo).
# Luego ejecuta esta celda.

input('Mueve el brazo sobre el objeto A y presiona Enter...')
pick_upper = mc.get_angles()
print(f'pick_upper = {pick_upper}')

In [ ]:
# Pose 3/5 — pick_grasp
# Baja el brazo hasta que el gripper toca el objeto A.
# Luego ejecuta esta celda.

input('Baja el brazo hasta tocar el objeto A y presiona Enter...')
pick_grasp = mc.get_angles()
print(f'pick_grasp = {pick_grasp}')

In [ ]:
# Pose 4/5 — place_upper
# Mueve el brazo hasta quedar SOBRE la zona de depósito B.
# Luego ejecuta esta celda.

input('Mueve el brazo sobre la zona B y presiona Enter...')
place_upper = mc.get_angles()
print(f'place_upper = {place_upper}')

In [ ]:
# Pose 5/5 — place_grasp
# Baja el brazo hasta la altura de depósito en zona B.
# Luego ejecuta esta celda.

input('Baja el brazo hasta la altura de depósito en B y presiona Enter...')
place_grasp = mc.get_angles()
print(f'place_grasp = {place_grasp}')

In [ ]:
# Verificación: todas las poses deben estar definidas
POSES = {
    'init_pose':   init_pose,
    'pick_upper':  pick_upper,
    'pick_grasp':  pick_grasp,
    'place_upper': place_upper,
    'place_grasp': place_grasp,
}

print('Resumen de poses capturadas:')
print(f'{"Pose":<14}  {"Ángulos [q1..q6]"}')
print('-' * 70)
todas_ok = True
for nombre, angulos in POSES.items():
    estado = 'OK' if angulos is not None else 'FALTA'
    if angulos is None:
        todas_ok = False
    print(f'  {nombre:<12}  {estado}  {[round(a,2) for a in angulos] if angulos else "None"}')

print()
if todas_ok:
    print('Pregunta 1: OK — todas las poses definidas.')
else:
    print('ATENCIÓN: falta capturar algunas poses.')

---
## Pregunta 2 — Ciclo de agarre con `send_angles` + `set_gripper_state`

**Enunciado:** Implementar el ciclo de agarre usando `send_angles` para mover el brazo y `set_gripper_state` para abrir/cerrar el gripper.

**Solución:** La secuencia del ciclo es una lista ordenada de poses y comandos de gripper. Cada elemento se ejecuta en orden:

```
init → pick_upper → pick_grasp → CLOSE → pick_upper → init
     → place_upper → place_grasp → OPEN → place_upper → init
```

In [ ]:
# Parámetros de gripper y movimiento
MOVE_SPEED    = 90    # se ajusta en Pregunta 3
GRIPPER_SPEED = 50
WAIT_MOVE     = 1.5   # se ajusta en Pregunta 3
WAIT_GRIP     = 1.0

# Secuencia completa del ciclo A → B
# Cada elemento: lista de ángulos  →  send_angles
#                'open'            →  set_gripper_state(0)
#                'close'           →  set_gripper_state(1)
SECUENCIA = [
    POSES['init_pose'],    #  1. reposo inicial
    POSES['pick_upper'],   #  2. sobre objeto A
    POSES['pick_grasp'],   #  3. bajar a A
    'close',               #  4. cerrar gripper (agarrar)
    POSES['pick_upper'],   #  5. subir con objeto
    POSES['init_pose'],    #  6. clearance intermedio
    POSES['place_upper'],  #  7. sobre zona B
    POSES['place_grasp'],  #  8. bajar a B
    'open',                #  9. abrir gripper (soltar)
    POSES['place_upper'],  # 10. subir sin objeto
    POSES['init_pose'],    # 11. reposo final
]


def ejecutar_paso(paso):
    """Ejecuta un paso de la secuencia."""
    if paso == 'close':
        mc.set_gripper_state(1, GRIPPER_SPEED)   # 1 = cerrar
        time.sleep(WAIT_GRIP)
    elif paso == 'open':
        mc.set_gripper_state(0, GRIPPER_SPEED)   # 0 = abrir
        time.sleep(WAIT_GRIP)
    else:
        mc.send_angles(paso, MOVE_SPEED)
        time.sleep(WAIT_MOVE)


print('Pregunta 2: OK — secuencia y funciones definidas.')
print(f'Pasos por ciclo: {len(SECUENCIA)}')
for i, paso in enumerate(SECUENCIA):
    desc = paso if isinstance(paso, str) else [round(a,1) for a in paso]
    print(f'  {i+1:>2}. {desc}')

---
## Pregunta 3 — Calibrar velocidades y tiempos de espera

**Enunciado:** Calibrar la velocidad de movimiento y los tiempos de espera para que los movimientos sean seguros y repetibles.

**Solución:** Se prueban tres velocidades distintas midiendo el tiempo real con `is_moving()`. Se elige la mayor velocidad sin error y se ajusta `WAIT_MOVE` automáticamente con un 10% de margen.

In [ ]:
velocidades_prueba = [50, 70, 90]

print('=' * 52)
print('CALIBRACIÓN — init_pose → pick_upper')
print(f'{"Velocidad":>10}  {"T real (s)":>12}  {"Estado":>8}')
print('-' * 52)

resultados_cal = []

for vel in velocidades_prueba:
    # Volver a init antes de cada prueba
    mc.send_angles(POSES['init_pose'], 30)
    time.sleep(2.5)

    t0     = time.time()
    estado = 'OK'
    try:
        mc.send_angles(POSES['pick_upper'], vel)
        time.sleep(0.4)
        for _ in range(50):           # espera hasta que deje de moverse
            if mc.is_moving() == 0:
                break
            time.sleep(0.2)
    except Exception as e:
        estado = 'ERR'

    t_real = round(time.time() - t0, 2)
    resultados_cal.append((vel, t_real, estado))
    print(f'{vel:>10}  {t_real:>12.2f}  {estado:>8}')

print('=' * 52)

# Elegir la mayor velocidad sin error
ok = [(v, t) for v, t, e in resultados_cal if e == 'OK']
if ok:
    MOVE_SPEED = max(v for v, t in ok)
    t_ref      = next(t for v, t in ok if v == MOVE_SPEED)
    WAIT_MOVE  = round(t_ref * 1.1, 1)   # +10% de margen

print(f'\nVelocidad seleccionada : {MOVE_SPEED}')
print(f'WAIT_MOVE ajustado     : {WAIT_MOVE}s')
print('Pregunta 3: OK — velocidad y tiempos calibrados.')

# Volver a reposo
mc.send_angles(POSES['init_pose'], 30)
time.sleep(2.5)

---
## Pregunta 5 — Gestionar errores de comunicación

**Enunciado:** Gestionar errores de comunicación con el robot mediante reintentos y timeouts.

**Solución:** `send_with_retry` envuelve cualquier llamada serial. Si falla, lo intenta hasta `MAX_RETRIES` veces con 1 segundo de espera. Si todos fallan, lanza `RuntimeError` para que el ciclo lo capture sin detener el programa.

In [ ]:
MAX_RETRIES = 3

def send_with_retry(func, *args):
    """
    Ejecuta func(*args) con hasta MAX_RETRIES intentos.
    Espera 1s entre reintentos.
    Lanza RuntimeError si todos los intentos fallan.
    """
    for intento in range(1, MAX_RETRIES + 1):
        try:
            return func(*args)
        except Exception as e:
            print(f'    [REINTENTO {intento}/{MAX_RETRIES}] {func.__name__}: {e}')
            time.sleep(1.0)
    raise RuntimeError(f'Fallo tras {MAX_RETRIES} intentos: {func.__name__}')


def ejecutar_paso_seguro(paso):
    """Igual que ejecutar_paso pero usando send_with_retry."""
    if paso == 'close':
        send_with_retry(mc.set_gripper_state, 1, GRIPPER_SPEED)
        time.sleep(WAIT_GRIP)
    elif paso == 'open':
        send_with_retry(mc.set_gripper_state, 0, GRIPPER_SPEED)
        time.sleep(WAIT_GRIP)
    else:
        send_with_retry(mc.send_angles, paso, MOVE_SPEED)
        time.sleep(WAIT_MOVE)


def run_ciclo(num):
    """
    Ejecuta un ciclo completo A→B con manejo de errores.
    Retorna dict con resultado y paso donde falló (si aplica).
    """
    t0          = time.time()
    paso_fallo  = None

    for i, paso in enumerate(SECUENCIA):
        etiqueta = paso if isinstance(paso, str) else f'pose_{i+1}'
        print(f'    Paso {i+1:>2}/{len(SECUENCIA)}: {etiqueta}')
        try:
            ejecutar_paso_seguro(paso)
        except RuntimeError as e:
            print(f'    ERROR: {e}')
            paso_fallo = i + 1
            try:
                mc.send_angles(POSES['init_pose'], 30)  # recuperar
            except Exception:
                pass
            break

    return {
        'ciclo':       num,
        'exito':       paso_fallo is None,
        'paso_fallo':  paso_fallo,
        'tiempo':      round(time.time() - t0, 2),
    }


print('Pregunta 5: OK — send_with_retry y run_ciclo definidos.')

---
## Prueba unitaria — 1 ciclo de verificación

Ejecuta 1 ciclo para verificar poses y gripper antes de los 5 consecutivos.

In [ ]:
print('=== PRUEBA UNITARIA (1 ciclo) ===')

mc.send_angles(POSES['init_pose'], 30)
mc.set_gripper_state(0, GRIPPER_SPEED)
time.sleep(2)

r = run_ciclo(0)
print()
print(f'Resultado : {"OK" if r["exito"] else "FALLO"}')
print(f'Tiempo    : {r["tiempo"]}s')
if not r['exito']:
    print(f'Falló en paso {r["paso_fallo"]} — revisar pose o conexión')

---
## Pregunta 4 — Ejecutar 5 ciclos consecutivos y registrar tasa de éxito

**Enunciado:** Ejecutar 5 ciclos consecutivos sin intervención humana y registrar la tasa de éxito.

**Solución:** Loop de 5 iteraciones sobre `run_ciclo()`. Cada resultado se guarda en `log`. Si un ciclo falla, el robot regresa a `init_pose`, espera 3s y continúa con el siguiente sin detener el programa.

In [ ]:
N_CICLOS = 5
log      = []

mc.send_angles(POSES['init_pose'], 30)
mc.set_gripper_state(0, GRIPPER_SPEED)
time.sleep(2)

print('=' * 55)
print(f'INICIO {N_CICLOS} CICLOS — {datetime.now().strftime("%H:%M:%S")}')
print('=' * 55)

for n in range(1, N_CICLOS + 1):
    print(f'\n--- Ciclo {n}/{N_CICLOS} ---')
    resultado = run_ciclo(n)
    log.append(resultado)

    estado = 'OK' if resultado['exito'] else f'FALLO (paso {resultado["paso_fallo"]})'
    print(f'  -> {estado}  ({resultado["tiempo"]}s)')

    if not resultado['exito']:
        time.sleep(3)   # pausa antes del siguiente ciclo

print('\n' + '=' * 55)
print('FIN DE CICLOS')
print('=' * 55)

---
## Reporte de métricas

In [ ]:
n_ok   = sum(1 for r in log if r['exito'])
tasa   = n_ok / len(log) * 100 if log else 0
t_prom = sum(r['tiempo'] for r in log) / len(log) if log else 0

print('=' * 55)
print('REPORTE FINAL — P5 Control de Trayectorias')
print('=' * 55)
print(f'{"Ciclo":<8} {"Resultado":<12} {"Tiempo (s)":<12} {"Paso fallo"}')
print('-' * 55)
for r in log:
    res  = 'OK' if r['exito'] else 'FALLO'
    paso = '-' if r['exito'] else str(r['paso_fallo'])
    print(f'  {r["ciclo"]:<6} {res:<12} {r["tiempo"]:<12} {paso}')
print('=' * 55)
print(f'Exitosos     : {n_ok}/{len(log)}  ({tasa:.0f}%)')
print(f'Tiempo prom. : {t_prom:.1f}s/ciclo')
print()
veredicto = 'APROBADO' if tasa >= 80 else 'REQUIERE AJUSTE'
print(f'Resultado: {veredicto}  (criterio >= 80%)')
print('=' * 55)